# Módulo 2: Filtro Bayesiano Discreto — Teoría

**Módulo**: `2` | **Tipo**: TEÓRICO | **Fecha**: 2026-05-20

> **Prerequisitos**: Módulo 1 completo (Teorema de Bayes, distribuciones Bernoulli/Gaussiana, ciclo prior–posterior).

## Objetivos de aprendizaje

Al finalizar este módulo el estudiante será capaz de:

1. **Formular** el modelo de espacio de estados discreto con variables ocultas $x_k$ y observaciones $z_k$.
2. **Derivar** la ecuación de Chapman-Kolmogorov como el paso de predicción del filtro bayesiano.
3. **Aplicar** la regla de Bayes para el paso de actualización (corrección) y justificar la normalización.
4. **Calcular** manualmente todos los pasos del filtro en un ejemplo de localización binaria (puerta).
5. **Implementar** el filtro bayesiano discreto en NumPy para seguimiento en una grilla 1D.

In [1]:
import sys
import numpy as np
import scipy
from scipy import stats

np.random.seed(42)
assert sys.version_info >= (3, 12), f"Python 3.12+ requerido, tienes {sys.version}"
print(f"Python {sys.version_info.major}.{sys.version_info.minor}  |  NumPy {np.__version__}  |  SciPy {scipy.__version__}  ✓")

Python 3.14  |  NumPy 2.4.6  |  SciPy 1.17.1  ✓


## Intuición — El robot que no sabe dónde está

Imagina un robot en un pasillo de longitud N. El robot sabe que puede estar en cualquiera de las N posiciones, pero no las puede observar directamente. Solo tiene acceso a un sensor ruidoso que mide la distancia a la pared más cercana.

El robot mantiene una **distribución de creencia** (en inglés: *belief*): un vector $\mathbf{b}$ donde $b[i] = P(x = i \mid z_{1:k})$ es la probabilidad de que esté en la posición $i$ dado todo lo que ha observado hasta el paso $k$.

En cada paso de tiempo, el robot hace dos cosas:

1. **Predice** (*predict*): sabe que se movió aproximadamente un paso hacia adelante, pero con incertidumbre. Su creencia se "difumina" según el modelo de movimiento.
2. **Corrige** (*update*): recibe una nueva medición del sensor. Las posiciones compatibles con la medición ganan probabilidad; las incompatibles la pierden.

Este proceso es exactamente el **Filtro Bayesiano Discreto**: el ciclo predict–update sobre un vector discreto de probabilidades.

## 1. Modelo de espacio de estados discreto

### Variables

| Símbolo | Nombre | Descripción |
|---------|--------|-------------|
| $x_k \in \{x_1, \ldots, x_N\}$ | **Estado oculto** | Posición real del agente en el paso $k$. No observable. |
| $z_k$ | **Observación** | Medición del sensor en el paso $k$. Ruidosa. |
| $b_k[i] = P(x_k = x_i \mid z_{1:k})$ | **Belief** (distribución de creencia) | Probabilidad posterior sobre el estado en el paso $k$. |

### Suposiciones del modelo

1. **Propiedad de Markov**: El estado futuro $x_{k+1}$ depende solo del estado actual $x_k$, no del historial completo:

$$P(x_{k+1} \mid x_k, x_{k-1}, \ldots, x_0) = P(x_{k+1} \mid x_k)$$

2. **Independencia condicional de las observaciones**: La observación $z_k$ depende solo del estado actual $x_k$:

$$P(z_k \mid x_k, x_{k-1}, \ldots, z_{k-1}) = P(z_k \mid x_k)$$

Bajo estas suposiciones, el modelo define un **proceso oculto de Markov** (*HMM* — Hidden Markov Model).

## 2. Ecuación de Chapman-Kolmogorov — Derivación del paso de predicción

Queremos calcular $\overline{b}_{k+1}[x']$ — la creencia en el paso $k+1$ *antes* de ver la nueva observación $z_{k+1}$.

### Paso 1: Ley de probabilidad total

Sumamos sobre todos los estados posibles del paso anterior:

$$\overline{b}_{k+1}[x'] = P(x_{k+1} = x' \mid z_{1:k}) = \sum_{x} P(x_{k+1} = x' \mid x_k = x,\; z_{1:k}) \cdot P(x_k = x \mid z_{1:k})$$

### Paso 2: Aplicar propiedad de Markov

Por la propiedad de Markov, $P(x_{k+1} \mid x_k, z_{1:k}) = P(x_{k+1} \mid x_k)$:

$$\overline{b}_{k+1}[x'] = \sum_{x} \underbrace{P(x_{k+1} = x' \mid x_k = x)}_{T[x', x]} \cdot \underbrace{P(x_k = x \mid z_{1:k})}_{b_k[x]}$$

### Paso 3: Forma matricial

Definiendo la **matriz de transición** $T$ con $T[i,j] = P(x_{k+1} = x_i \mid x_k = x_j)$:

$$\boxed{\overline{\mathbf{b}}_{k+1} = T \cdot \mathbf{b}_k}$$

Esta es la **ecuación de Chapman-Kolmogorov** en su forma discreta. Es una convolución del belief con el modelo de movimiento.

In [2]:
# Verificación numérica — ecuación de Chapman-Kolmogorov
N = 5  # 5 posiciones discretas

# Modelo de transición: el agente se mueve +1 con prob 0.7, se queda con 0.3
T = np.zeros((N, N), dtype=np.float64)
for j in range(N):
    j_next = min(j + 1, N - 1)  # mover +1, con límite en el borde
    T[j_next, j] = 0.7
    T[j, j] += 0.3

print("Matriz de transición T (columnas suman 1):")
print(np.round(T, 2))
print(f"Sumas de columnas: {T.sum(axis=0).round(2)}")
print()

# Belief inicial: concentrado en posición 1
b = np.zeros(N, dtype=np.float64)
b[1] = 1.0

print("Belief inicial:")
print(f"  {b}  (concentrado en x=1)")
print()

# Predicción Chapman-Kolmogorov: b_bar = T @ b
b_bar = T @ b
print("Belief tras predicción (Chapman-Kolmogorov):")
print(f"  {b_bar}  (masa se difunde)")
assert abs(b_bar.sum() - 1.0) < 1e-10, "Suma del belief no es 1"
print(f"Suma del belief: {b_bar.sum():.6f}  ✓")

Matriz de transición T (columnas suman 1):
[[0.3 0.  0.  0.  0. ]
 [0.7 0.3 0.  0.  0. ]
 [0.  0.7 0.3 0.  0. ]
 [0.  0.  0.7 0.3 0. ]
 [0.  0.  0.  0.7 1. ]]
Sumas de columnas: [1. 1. 1. 1. 1.]

Belief inicial:
  [0. 1. 0. 0. 0.]  (concentrado en x=1)

Belief tras predicción (Chapman-Kolmogorov):
  [0.  0.3 0.7 0.  0. ]  (masa se difunde)
Suma del belief: 1.000000  ✓


## 3. Regla de Bayes — Derivación del paso de actualización

Tenemos $\overline{b}_{k+1}[x']$ (belief predicha) y recibimos la observación $z_{k+1}$. Queremos $b_{k+1}[x']$.

### Paso 1: Aplicar regla de Bayes

$$b_{k+1}[x'] = P(x_{k+1} = x' \mid z_{k+1}, z_{1:k}) = \frac{P(z_{k+1} \mid x_{k+1} = x') \cdot P(x_{k+1} = x' \mid z_{1:k})}{P(z_{k+1} \mid z_{1:k})}$$

### Paso 2: Sustituir el prior (belief predicha)

$$b_{k+1}[x'] = \frac{P(z_{k+1} \mid x') \cdot \overline{b}_{k+1}[x']}{P(z_{k+1} \mid z_{1:k})}$$

### Paso 3: Calcular el denominador (normalización)

$$P(z_{k+1} \mid z_{1:k}) = \sum_{x'} P(z_{k+1} \mid x') \cdot \overline{b}_{k+1}[x'] = \eta^{-1}$$

### Resultado: Paso de corrección

$$\boxed{b_{k+1}[x'] = \eta \cdot P(z_{k+1} \mid x') \cdot \overline{b}_{k+1}[x']}$$

donde $\eta = \left[\sum_{x'} P(z_{k+1} \mid x') \cdot \overline{b}_{k+1}[x']\right]^{-1}$ es la constante de normalización.

In [3]:
# Verificación numérica — paso de actualización bayesiana
# Continuando el ejemplo anterior (N=5, movimiento +1)

# Observación: el sensor dice z=2 (está en o cerca de la posición 2)
# Modelo de sensor: P(z | x) = N(z; x, sigma=0.8)
sigma_sensor = 0.8
z_obs = 2.0
grilla = np.arange(N, dtype=np.float64)  # posiciones 0, 1, 2, 3, 4

# Calcular likelihood P(z | x) para cada estado
likelihood = stats.norm.pdf(grilla, loc=z_obs, scale=sigma_sensor)
print(f"Observación z = {z_obs}")
print(f"Likelihood P(z | x) para cada posición:")
for i, (pos, lik) in enumerate(zip(grilla, likelihood)):
    print(f"  P(z={z_obs} | x={pos:.0f}) = {lik:.4f}")

print()
print(f"Belief predicha b_bar: {b_bar.round(3)}")

# Actualización: b = η · P(z|x) · b_bar
unnorm = likelihood * b_bar
eta_inv = unnorm.sum()
b_updated = unnorm / eta_inv

print(f"Después de corrección con z={z_obs}:")
print(f"  Belief actualizado: {b_updated.round(4)}")
print(f"  MAP (argmax): x = {grilla[np.argmax(b_updated)]:.0f}")
print(f"  Suma: {b_updated.sum():.6f}  ✓")

Observación z = 2.0
Likelihood P(z | x) para cada posición:
  P(z=2.0 | x=0) = 0.0219
  P(z=2.0 | x=1) = 0.2283
  P(z=2.0 | x=2) = 0.4987
  P(z=2.0 | x=3) = 0.2283
  P(z=2.0 | x=4) = 0.0219

Belief predicha b_bar: [0.  0.3 0.7 0.  0. ]
Después de corrección con z=2.0:
  Belief actualizado: [0.    0.164 0.836 0.    0.   ]
  MAP (argmax): x = 2
  Suma: 1.000000  ✓


## Ejemplo numérico paso a paso

### El problema de la puerta

Consideramos un robot que se mueve en un pasillo binario de **3 posiciones**: {0, 1, 2}.

El robot observa si hay una "puerta" en su posición. El modelo es:

- **Prior**: uniforme, $b_0 = [1/3, 1/3, 1/3]$
- **Modelo de transición**: el robot se mueve de $x$ a $x+1$ con probabilidad 1 (determinista, con límite en el borde)
- **Modelo de sensor**: $P(z=\text{puerta} \mid x) = [0.8, 0.1, 0.6]$ para $x \in \{0, 1, 2\}$

Calculamos el filtro para 5 pasos con observaciones: [puerta, no-puerta, puerta, puerta, no-puerta].

In [4]:
# Ejemplo de la puerta — 3 posiciones, derivación completa
N_puerta = 3
grilla_p = np.array([0.0, 1.0, 2.0], dtype=np.float64)

# Prior uniforme
b = np.array([1/3, 1/3, 1/3], dtype=np.float64)

# Modelo de transición determinista: x → x+1 (con límite en borde)
T_puerta = np.array([
    [0.0, 0.0, 0.0],   # Para llegar a x'=0: desde x=0 con prob 0 (no puede retroceder)
    [1.0, 0.0, 0.0],   # Para llegar a x'=1: desde x=0 con prob 1
    [0.0, 1.0, 1.0],   # Para llegar a x'=2: desde x=1 con prob 1, o desde x=2 (límite)
], dtype=np.float64)
# Nota: T[:,j] son las probabilidades de llegar a cada estado desde j
# Corregir: si está en x=2, se queda en 2
T_puerta = np.array([
    [0.0, 0.0, 0.0],
    [1.0, 0.0, 0.0],
    [0.0, 1.0, 1.0],
], dtype=np.float64)

# Modelo de sensor: P(z=puerta | x) para x ∈ {0,1,2}
p_puerta_dado_x = np.array([0.8, 0.1, 0.6], dtype=np.float64)  # probabilidad de observar puerta

# Secuencia de observaciones (1=puerta, 0=no-puerta)
observaciones_p = [1, 0, 1, 1, 0]

print(f"{'Paso':>4} | {'Obs':>5} | {'b[0]':>8} | {'b[1]':>8} | {'b[2]':>8} | {'MAP':>4}")
print("-" * 55)
print(f"{'0':>4} | {'---':>5} | {b[0]:>8.4f} | {b[1]:>8.4f} | {b[2]:>8.4f} | {np.argmax(b):>4}")

for t, z in enumerate(observaciones_p, start=1):
    # Paso 1: Predicción (Chapman-Kolmogorov)
    b_bar = T_puerta @ b

    # Paso 2: Corrección (regla de Bayes)
    if z == 1:  # observó puerta
        likelihood_p = p_puerta_dado_x
    else:       # observó no-puerta
        likelihood_p = 1 - p_puerta_dado_x

    b = likelihood_p * b_bar
    b = b / b.sum()  # normalizar

    obs_str = "puerta" if z == 1 else "no-pta"
    print(f"{t:>4} | {obs_str:>5} | {b[0]:>8.4f} | {b[1]:>8.4f} | {b[2]:>8.4f} | {np.argmax(b):>4}")

assert abs(b.sum() - 1.0) < 1e-10, "Belief final no normalizado"
print()
print("✓ Tabla completada — belief correctamente normalizado en cada paso.")

Paso |   Obs |     b[0] |     b[1] |     b[2] |  MAP
-------------------------------------------------------
   0 |   --- |   0.3333 |   0.3333 |   0.3333 |    0
   1 | puerta |   0.0000 |   0.0769 |   0.9231 |    2
   2 | no-pta |   0.0000 |   0.0000 |   1.0000 |    2
   3 | puerta |   0.0000 |   0.0000 |   1.0000 |    2
   4 | puerta |   0.0000 |   0.0000 |   1.0000 |    2
   5 | no-pta |   0.0000 |   0.0000 |   1.0000 |    2

✓ Tabla completada — belief correctamente normalizado en cada paso.


## 4. Normalización de la distribución de creencia

La normalización es esencial: la suma $\sum_x b_k[x]$ debe ser 1 en todo momento.

**¿Por qué puede romperse?** Si multiplicamos element-wise por el likelihood (valores < 1), la suma disminuye y ya no es una distribución válida.

**Solución**: dividir siempre entre la suma del vector no-normalizado:

$$b_{k+1}[x'] = \frac{P(z_{k+1} \mid x') \cdot \overline{b}_{k+1}[x']}{\sum_{x''} P(z_{k+1} \mid x'') \cdot \overline{b}_{k+1}[x'']}$$

**Caso especial**: Si $\sum_{x'} P(z_{k+1} \mid x') \cdot \overline{b}_{k+1}[x'] \approx 0$, el belief "colapsó". Esto ocurre cuando la observación es imposible dada la creencia actual — indica un error en el modelo o en la observación.

**Verificación continua**:
```python
assert abs(belief.sum() - 1.0) < 1e-6, "Belief no normalizado"
```

## 5. Extensión a grilla N-dimensional

Para seguimiento en una grilla 1D de $N$ posiciones, el mismo algoritmo aplica:

| Componente | Descripción |
|-----------|-------------|
| $\mathbf{b} \in \mathbb{R}^N$ | Vector de creencia (belief), $b[i] = P(x_k = i \mid z_{1:k})$ |
| $T \in \mathbb{R}^{N \times N}$ | Matriz de transición. $T[i,j] = P(x_{k+1}=i \mid x_k=j)$. Columnas suman 1. |
| $\mathbf{l}(z) \in \mathbb{R}^N$ | Vector de likelihood. $l[i] = P(z_k \mid x_k=i)$ |

**Algoritmo completo**:
```
Inicializar: b = prior_uniforme (b[i] = 1/N)
Para cada paso k:
    1. Predicción:  b_bar = T @ b
    2. Corrección:  b = l(z_k) * b_bar
                    b = b / b.sum()
```

**Complejidad**: $O(N^2)$ por paso (dominado por la multiplicación matricial $T \cdot b$).
Para grillas grandes, si $T$ es un kernel local (banda), se puede usar convolución $O(N \log N)$.

## Errores comunes

### Error 1: No normalizar el belief tras la corrección

**Síntoma**: El belief acumula error numérico y deja de sumar 1 tras varios pasos. Las probabilidades calculadas son incorrectas. Con likelihood < 1 en todos los estados, el belief se "encoge" y eventualmente todos los valores se acercan a 0.

**Diagnóstico**: El estudiante implementa `b = likelihood * b_bar` pero olvida dividir entre `b.sum()`. Tras $k$ pasos, `b.sum()` ≈ $(0.5)^k$ → 0.

**Solución**: Normalizar siempre después de cada corrección:
```python
b = likelihood * b_bar
b = b / b.sum()   # ← OBLIGATORIO
assert abs(b.sum() - 1.0) < 1e-8
```

---

### Error 2: Transponer incorrectamente la matriz de transición

**Síntoma**: El belief se mueve en la dirección contraria a la esperada, o se concentra en estados que el agente no debería poder alcanzar.

**Diagnóstico**: La convención correcta es $T[i,j] = P(x'=i \mid x=j)$ — las **columnas** son distribuciones sobre el estado siguiente dado el estado actual. Si el estudiante define $T[i,j] = P(x'=j \mid x=i)$ (convención inversa) y aplica $T \cdot b$, obtiene el resultado transpuesto.

**Solución**: Verificar siempre que cada **columna** de $T$ sume 1 (es una distribución de probabilidad válida sobre el estado siguiente):
```python
assert np.allclose(T.sum(axis=0), 1.0), "Columnas de T deben sumar 1"
```

## Evaluación

**Problema M2-E1**: Sea el modelo de la puerta (3 posiciones) del Ejemplo numérico. Calcula a mano (o en Python) el belief después de 2 pasos con observaciones [no-puerta, puerta]. Usa los mismos parámetros del ejemplo.

**Criterio**: $|b_2[i] - b_{\text{ref}}[i]| < 10^{-6}$ para $i \in \{0,1,2\}$.

**Problema M2-E2**: Demuestra que si el modelo de transición es la matriz identidad $T = I$ (el agente no se mueve), el filtro bayesiano discreto se reduce al modelo de actualización bayesiana del Módulo 1 (multiplicación del prior por la likelihood, seguida de normalización).

**Problema M2-E3**: ¿Cuántos pasos de actualización se necesitan para que la entropía del belief $H[b] = -\sum_x b[x] \log b[x]$ baje por debajo de $\log(N/2)$ en el ejemplo de seguimiento en grilla (N=20)?

## ¿Qué aprendiste?

Aprendimos que…

1. El filtro bayesiano discreto es la extensión natural del Teorema de Bayes a **estados dinámicos**: en lugar de actualizar la creencia sobre un parámetro estático, actualizamos la creencia sobre el estado de un sistema que evoluciona en el tiempo.
2. La **predicción** (Chapman-Kolmogorov) es una convolución del belief con el modelo de movimiento — propaga la incertidumbre hacia adelante en el tiempo.
3. La **corrección** (regla de Bayes) reduce la incertidumbre multiplicando el belief por la likelihood de la observación y normalizando.
4. La **normalización** no es opcional: sin ella, el belief deja de ser una distribución de probabilidad válida en pocos pasos.
5. El ciclo predict–update es el mismo en todos los filtros del curso: lo que cambia es la forma del espacio de estados (discreto aquí, continuo en Kalman, muestral en partículas).

## ¿Qué sigue?

El siguiente sub-módulo es el **Módulo 3a: Filtro de Kalman Lineal**.

En el filtro bayesiano discreto, el espacio de estados es un vector finito de N celdas. El costo computacional por paso es $O(N^2)$, lo que lo hace impracticable para estados continuos o de alta dimensión.

El Filtro de Kalman resuelve esto asumiendo que el **espacio de estados es continuo y Gaussiano**: en lugar de propagar un vector de N probabilidades, propagamos solo la **media** $\hat{x}_k$ y la **covarianza** $P_k$ de una distribución Gaussiana. Esto reduce el costo a $O(n^3)$ donde $n$ es la dimensión del estado (típicamente 2-6), independientemente del rango de valores posibles.